## Prepare Env

In [ ]:
%pip install boto3 pyspark delta-spark python-dotenv

In [ ]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

## Ingestion
### 1. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("JsonToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
file_path = "s3a://warehouse/files/opensanctions.file/entities.ftm.json"
delta_table_path = "s3a://warehouse/bronze/opensanctions_entities.delta"

In [ ]:
# Read file into a DataFrame
df = spark.read.json(file_path)

In [ ]:
df.show()

In [ ]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert

Read delta table and discovery data

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("JsonToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
delta_table_path = "s3a://warehouse/bronze/opensanctions_entities.delta"

In [ ]:
# Read Delta table
df = spark.read.format("delta").load(delta_table_path)

In [ ]:
df.show()